# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d02_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # pile_settlement train
    shutil.copy(glob.glob("/kaggle/input/**/d02_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # pile_settlement test
    print("Kaggle: pile_settlement data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: pile_settlement data copied to ./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795364002.596017      34 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795364016.872111      34 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [4]:
# Go to find & replace button and replace (pile_settlement_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['D', 'L', 'N30_Ava', 'Cu_Ava', 'N30_Base', 'Cu_Base', 'UL', 'Q ']
TARGET_LABEL = 'Settlement (S_exp)'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (371, 9)
First 5 rows of training data:
      D     L    N30_Ava      Cu_Ava  N30_Base  Cu_Base  UL       Q      S_exp 
0  1.0  43.5   3.317241  105.408046        50      0.0   1     0.00   0.000000
1  0.8  24.0   8.875000   17.708333         0     50.0   1   675.00   0.691250
2  0.8  24.0   5.032258   20.000000         0     40.0   1  1012.50   0.870417
3  1.2  70.0  18.214286   14.142857        50      0.0   2  7485.15  19.161300
4  1.0  34.0   6.741176   89.117647        50      0.0   2  3007.59   6.552300

Shape of test data: (186, 9)
First 5 rows of test data:
      D     L    N30_Ava     Cu_Ava  N30_Base  Cu_Base  UL        Q     S_exp 
0  0.8  24.0   8.875000  17.708333         0     50.0   2  1687.500  3.295000
1  1.0  38.0  26.631579  49.539474        50      0.0   2  2250.000  4.595960
2  0.8  24.0   5.032258  20.000000         0     40.0   1  1687.500  1.721250
3  1.0  35.5  28.400000  36.676056        50      0.0   1   741.259  0.441964
4  1.2  57.0

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (371, 8)
Shape of y_train: (371,)
Shape of X_test: (186, 8)
Shape of y_test: (186,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.11501862  0.7026739  -1.39285389  2.03926221  0.47278688 -0.36847246
  -0.88040878 -0.93211799]
 [-1.33640677 -0.87840793 -0.78031847 -0.77862641 -1.76108898  0.85111254
  -0.88040878 -0.72895693]
 [-1.33640677 -0.87840793 -1.2038373  -0.70499262 -1.76108898  0.60719554
  -0.88040878 -0.6273764 ]
 [ 1.566444    2.85132356  0.24898912 -0.89318909  0.47278688 -0.36847246
   1.13583602  1.32075756]
 [ 0.11501862 -0.06759674 -1.0154929   1.5158337   0.47278688 -0.36847246
   1.13583602 -0.02689552]]

First five rows of normalized X_test:
[[-1.33640677 -0.87840793 -0.78031847 -0.77862641 -1.76108898  0.85111254
   1.13583602 -0.42421534]
 [ 0.11501862  0.25672774  1.1766814   0.2441433   0.47278688 -0.36847246
   1.13583602 -0.25491446]
 [-1.33640677 -0.87840793 -1.2038373  -0.70499262 -1.76108898  0.60719554
  -0.88040878 -0.42421534]
 [ 0.11501862  0.05402494  1.37158379 -0.16917252  0.47278688 -0.36847246
  -0.88040878 -0.70901434]
 [ 1.566444

# **Functions:**

In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [12]:
# FIX Q2: quantile crossing. Each quantile level is a separate model, so for some rows the
# predictions can come out in the wrong order (e.g. q0.10 < q0.05). Before any metric we sort
# each row's quantile predictions (monotone rearrangement, Chernozhukov et al. 2010) and record
# how many rows crossed, for the Excel report.
quantile_crossing_report = {}

def rearrange_quantiles(predictions_df, tag=None):
    q_cols = sorted(c for c in predictions_df.columns if c != 'Actual')
    values = predictions_df[q_cols].to_numpy(dtype=float)
    crossed = (np.diff(values, axis=1) < 0).any(axis=1)
    if tag is not None:
        quantile_crossing_report[tag] = {'rows': int(len(values)), 'rows_crossing': int(crossed.sum()),
                                         'pct_crossing': round(100.0 * crossed.mean(), 2)}
    out = predictions_df.copy()
    out[q_cols] = np.sort(values, axis=1)
    return out


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def quantile_regression_print(predictions_df, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Extract actual values
    actual_values = predictions_df['Actual'].values

    # Debugging: Print the first few actual values
    print("First few actual values:", actual_values[:5])

    # Calculate coverage for the specific interval (0.05 to 0.95)
    lower_quantile = 0.05
    median_quantile = 0.5
    upper_quantile = 0.95

    lower_preds = predictions_df[lower_quantile].values
    median_preds = predictions_df[median_quantile].values
    upper_preds = predictions_df[upper_quantile].values

    in_interval = ((actual_values >= lower_preds) & (actual_values <= upper_preds))
    coverage = in_interval.mean()
    print(f"Coverage of 90% prediction interval: {coverage * 100:.2f}%")

    # Create a new Excel workbook and add a worksheet
    wb = Workbook()
    ws = wb.active
    ws.title = "quantile_regression_print"

    # Plot actual vs. all predicted quantiles
    num_points = len(predictions_df)
    indices = np.arange(num_points)

    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values[:num_points], label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')

    # Loop over all columns assumed to be quantile predictions
    for column in predictions_df.columns:
        if column != 'Actual':  # Skip the actual values column
            plt.plot(indices, predictions_df[column][:num_points],
                     label=f'Predicted Quantile {column}', linestyle='--')

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title(f'Actual vs. Predicted Quantile {TARGET_LABEL}')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A1')

    # Plot actual values and specific prediction intervals (0.05, 0.5, 0.95)
    plt.figure(figsize=(12, 6))
    plt.plot(indices, actual_values, label=f'Actual {TARGET_LABEL}', marker='o', linestyle='-', color='black')  # Connect points with a line
    plt.plot(indices, median_preds, label=f'Median Prediction ({median_quantile})', marker='x', linestyle='-', color='blue')
    plt.fill_between(
        indices,
        lower_preds,
        upper_preds,
        color='blue',
        alpha=0.5,
        label=f'Prediction Interval ({lower_quantile}-{upper_quantile})'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title(f'Actual {TARGET_LABEL} with Prediction Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1, 1))

    # Save the second plot to another BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Insert the second image into the Excel sheet
    img = Image(img_data)
    ws.add_image(img, 'A30')  # Adjust the cell position as needed

    # Save the workbook
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluation_features_print(predictions_df, X_test, file_path):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    # Ensure X_test is a DataFrame
    if isinstance(X_test, np.ndarray):
        X_test = pd.DataFrame(X_test, columns=feature_names)
    X_test.columns = feature_names

    # Get quantile columns from predictions_df except 'Actual'
    quantile_columns = [col for col in predictions_df.columns if col != 'Actual']

    # Check if the Excel file already exists
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="New Evaluation Features")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "evaluation_features_print"

    image_row = 1  # Where to start images

    for feature_to_plot in feature_names:
        feature_values = X_test[feature_to_plot]
        plot_df = predictions_df.copy()
        plot_df[feature_to_plot] = feature_values.values

        # Sort by current feature to plot lines well
        plot_df = plot_df.sort_values(by=feature_to_plot)

        # Begin Matplotlib plot
        plt.figure(figsize=(6, 4))
        # Plot each quantile as a line
        for quantile in quantile_columns:
            plt.plot(plot_df[feature_to_plot], plot_df[quantile], label=f"{quantile}")

        # Plot actual values as scatter
        plt.scatter(plot_df[feature_to_plot], plot_df['Actual'], color='black', s=12, label=f"Actual {TARGET_LABEL}")

        plt.title(f"Predicted Quantiles vs. {feature_to_plot}")
        plt.xlabel(feature_to_plot)
        plt.ylabel(f"{TARGET_LABEL}")
        plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
        plt.tight_layout()

        # Save figure to a BytesIO stream
        img_bytes = BytesIO()
        img_bytes
        plt.savefig(img_bytes, format="png")
        plt.close()
        img_bytes.seek(0)

        img = Image(img_bytes)
        img.anchor = f"A{image_row}"
        ws.add_image(img)

        image_row += 20  # Adjust as you like

    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def evaluate_uncertainity_matrix(predictions_df, quantiles, excel_file_path, model_name='Model'):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2

   # Extract the actual target values
    y_true = predictions_df['Actual']

    # Open existing Excel file
    wb = load_workbook(_ensure_excel_file(excel_file_path))

    # Create unique sheet names
    def get_unique_sheet_name(wb, base_name):
        sheet_name = base_name
        i = 1
        while sheet_name in wb.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        return sheet_name

    # Create separate sheets for plots and values
    plots_sheet_title = get_unique_sheet_name(wb, 'Plots')
    values_sheet_title = get_unique_sheet_name(wb, 'Values')

    ws_plots = wb.create_sheet(title=plots_sheet_title)
    ws_values = wb.create_sheet(title=values_sheet_title)
    # FIX Q2: how often the raw quantile predictions crossed (before sorting)
    _cr = quantile_crossing_report.get(excel_file_path, {})
    ws_values.append(['Rows with quantile crossing (before sorting)', _cr.get('rows_crossing'), 'of', _cr.get('rows'), 'rows', f"{_cr.get('pct_crossing')}%"])
    print(f"Quantile crossing before sorting: {_cr.get('rows_crossing')} of {_cr.get('rows')} rows ({_cr.get('pct_crossing')}%)")

    # Positions to place images
    image_positions = ['A1', 'A25', 'A49', 'A73']  # Adjust as needed

    ### 1. Validity / Coverage ###

    # Compute empirical coverage for each quantile
    empirical_quantiles = []
    for q in quantiles:
        coverage = (predictions_df[q] >= y_true).mean()
        empirical_quantiles.append(coverage)

    # Plot theoretical quantiles vs empirical quantiles
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=quantiles, y=quantiles, color="magenta", linestyle='--', linewidth=2, label="Ideal")
    sns.lineplot(x=quantiles, y=empirical_quantiles, color="blue", linewidth=2, label="Empirical")
    sns.scatterplot(x=quantiles, y=empirical_quantiles, color="blue", s=50)
    plt.xlabel("Nominal Quantile Levels")
    plt.ylabel("Empirical Quantile Levels")
    plt.title(f"Validity / Coverage - {model_name}")
    plt.legend()

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add the image to the plots worksheet
    ws_plots.add_image(img, image_positions[0])

    ### 2. Sharpness / Interval Length ###

    # Define coverage levels to evaluate
    coverage_levels = [0.6, 0.8, 0.9]
    average_interval_lengths = []

    # Compute average interval lengths for each coverage level
    for c in coverage_levels:
        lower_q = (1 - c) / 2
        upper_q = 1 - lower_q
        # Find the nearest quantiles in our quantiles list
        lower_quantile = min(quantiles, key=lambda x: abs(x - lower_q))
        upper_quantile = min(quantiles, key=lambda x: abs(x - upper_q))

        # Compute interval length for each observation
        interval_length = predictions_df[upper_quantile] - predictions_df[lower_quantile]
        avg_interval_length = interval_length.mean()
        average_interval_lengths.append(avg_interval_length)
        # Write interval length information to the values worksheet
        ws_values.append([f'Coverage level: {c*100:.0f}%', f'Interval: [{lower_quantile}, {upper_quantile}]', f'Average Interval Length: {avg_interval_length:.4f}'])

    # Plot coverage levels vs average interval lengths
    plt.figure(figsize=(10, 6))
    plt.plot([c*100 for c in coverage_levels], average_interval_lengths, marker='o')
    plt.xlabel('Coverage Level (%)')
    plt.ylabel('Average Interval Length')
    plt.title(f'Sharpness / Interval Length - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[1])

    ### 3. Negative Log-Likelihood (NLL) ###

    # Use the 5th and 95th percentiles to estimate standard deviation
    if 0.05 in quantiles and 0.95 in quantiles:
        z_lower = norm.ppf(0.05)  # z-score for 5% quantile (~ -1.6449)
        z_upper = norm.ppf(0.95)  # z-score for 95% quantile (~ +1.6449)

        # Extract quantile predictions
        q_lower = predictions_df[0.05]
        q_upper = predictions_df[0.95]

    elif 0.1 in quantiles and 0.9 in quantiles:
        z_lower = norm.ppf(0.1)  # z-score for 10% quantile (~ -1.2816)
        z_upper = norm.ppf(0.9)  # z-score for 90% quantile (~ +1.2816)

        # Extract quantile predictions
        q_lower = predictions_df[0.1]
        q_upper = predictions_df[0.9]
    else:
        raise ValueError("Required quantiles for NLL estimation not found in quantiles list.")

    # Use median as the mean estimate
    if 0.5 in quantiles:
        q_median = predictions_df[0.5]
    else:
        # If 0.5 quantile is not available, use the middle quantile
        q_median = predictions_df[quantiles[len(quantiles)/2]]

    # Estimate standard deviation for each observation
    std_estimates = (q_upper - q_lower) / (z_upper - z_lower)

    # Ensure standard deviations are positive and non-zero
    std_estimates = std_estimates.clip(lower=1e-6)

    # Extract mean estimates (median predictions)
    mean_estimates = q_median

    # Compute Negative Log-Likelihood for each observation
    nll = -norm.logpdf(y_true, loc=mean_estimates, scale=std_estimates)

    # Compute average NLL
    average_nll = nll.mean()
    # Write average NLL to the values worksheet
    ws_values.append(['Average Negative Log-Likelihood (NLL)', average_nll])

    # Write NLL values to the values worksheet
    ws_values.append(['NLL Values'])
    for value in nll:
        ws_values.append([value])

    # Plot ECDF of NLL values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(nll, color='blue', linewidth=2)
    plt.xlabel('Negative Log-Likelihood (NLL)')
    plt.ylabel('ECDF')
    plt.title(f'NLL ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[2])

    ### 4. Continuous Ranked Probability Score (CRPS) ###

    # Prepare ensemble of quantile predictions for each observation
    ensemble_predictions = predictions_df[quantiles].to_numpy()

    # Ensure quantile predictions are sorted for each observation
    ensemble_predictions.sort(axis=1)

    # Compute CRPS for each observation using ensemble predictions
    # FIX Q4: CRPS from a set of quantiles = 2 * integral of the pinball loss over tau.
    # Each quantile level gets a weight equal to the share of [0, 1] it represents
    # (levels are unevenly spaced). Was: ps.crps_ensemble, which treats the 13 quantiles as
    # 13 equally likely samples.
    _taus = np.array(quantiles, dtype=float)
    _edges = np.concatenate([[0.0], (_taus[1:] + _taus[:-1]) / 2.0, [1.0]])
    _w = np.diff(_edges)
    _err = np.asarray(y_true, dtype=float)[:, None] - ensemble_predictions
    _pinball = np.maximum(_taus * _err, (_taus - 1.0) * _err)
    crps_values = 2.0 * (_pinball * _w).sum(axis=1)

    # Compute average CRPS
    average_crps = crps_values.mean()
    # Write average CRPS to the values worksheet
    ws_values.append(['Average Continuous Ranked Probability Score (CRPS)', average_crps])

    # Write CRPS values to the values worksheet
    ws_values.append(['CRPS Values'])
    for value in crps_values:
        ws_values.append([value])

    # Plot ECDF of CRPS values
    plt.figure(figsize=(10, 6))
    sns.ecdfplot(crps_values, color='green', linewidth=2)
    plt.xlabel('Continuous Ranked Probability Score (CRPS)')
    plt.ylabel('ECDF')
    plt.title(f'CRPS ECDF - {model_name}')

    # Save plot to buffer
    buffer = io.BytesIO()
    buffer
    plt.savefig(buffer, format='png')
    plt.close()
    buffer.seek(0)

    # Create an Image object
    img = openpyxlImage(buffer)
    img.width = 640
    img.height = 480

    # Add image to the plots worksheet
    ws_plots.add_image(img, image_positions[3])

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    wb.save(_ensure_parent_dir(excel_file_path))

    # Return a dictionary of results
    results = {
        'empirical_quantiles': empirical_quantiles,
        'average_interval_lengths': average_interval_lengths,
        'coverage_levels': coverage_levels,
        'average_nll': average_nll,
        'average_crps': average_crps,
        'nll_values': nll,
        'crps_values': crps_values
    }

    return results

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def calibrate_and_plot_intervals(predictions_df, excel_file_path, alpha=0.1, max_samples=1000, random_seed=42):
    predictions_df = rearrange_quantiles(predictions_df, excel_file_path)  # FIX Q2
    # Extract necessary columns
    lower = predictions_df[0.05].fillna(0)
    upper = predictions_df[0.95].fillna(0)
    pred = predictions_df[0.5].fillna(0)
    labels = predictions_df['Actual'].fillna(0)

    # Problem setup
    total_samples = labels.shape[0]
    n = min(max_samples, total_samples)  # Ensure n does not exceed the number of samples

    # Ensure there are enough samples for both calibration and validation
    if total_samples <= n:
        n = total_samples // 2

    # Split the data into calibration and validation sets
    np.random.seed(random_seed)  # Set the random seed for reproducibility
    idx = np.array([1] * n + [0] * (total_samples - n)) > 0
    np.random.shuffle(idx)
    cal_labels, val_labels = labels[idx], labels[~idx]
    cal_upper, val_upper = upper[idx], upper[~idx]
    cal_lower, val_lower = lower[idx], lower[~idx]
    cal_pred, val_pred = pred[idx], pred[~idx]

    # Calculate uncertainty intervals
    cal_U = cal_upper - cal_lower
    val_U = val_upper - val_lower

    # Avoid division by zero by replacing zero intervals with a small number
    cal_U[cal_U == 0] = np.finfo(float).eps
    val_U[val_U == 0] = np.finfo(float).eps

    # Get scores
    cal_scores = np.abs(cal_pred - cal_labels) / cal_U

    # Debugging: Check for NaN in scores
    if np.isnan(cal_scores).any():
        print("NaN values found in cal_scores. Check data integrity.")

    # Get the score quantile
    qhat = np.quantile(cal_scores, np.ceil((n + 1) * (1 - alpha)) / n, interpolation='higher')

    # Deploy (output=lower and upper adjusted quantiles)
    prediction_sets = [val_pred - val_U * qhat, val_pred + val_U * qhat]

    # Calculate empirical coverage (before and after calibration)
    prediction_sets_uncalibrated = [val_lower, val_upper]
    empirical_coverage_uncalibrated = ((val_labels >= prediction_sets_uncalibrated[0]) & (val_labels <= prediction_sets_uncalibrated[1])).mean() * 100
    print(f"The empirical coverage before calibration is: {empirical_coverage_uncalibrated:.2f}%")

    empirical_coverage = ((val_labels >= prediction_sets[0]) & (val_labels <= prediction_sets[1])).mean() * 100
    print(f"The empirical coverage after calibration is: {empirical_coverage:.2f}%")

    # Create a DataFrame with all the necessary data
    data = pd.DataFrame({
        'Index': np.arange(len(val_labels)),
        'Uncalibrated Lower': val_lower,
        'Uncalibrated Upper': val_upper,
        'Calibrated Lower': prediction_sets[0],
        'Calibrated Upper': prediction_sets[1],
        'True Label': val_labels,
        'Predicted Value': val_pred
    })

    plt.figure(figsize=(14, 8))

    # Plot the uncalibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Uncalibrated Lower'],
        data['Uncalibrated Upper'],
        color='blue',
        alpha=0.5,
        label='Uncalibrated Prediction Interval'
    )

    # Plot the calibrated prediction intervals as a shaded area
    plt.fill_between(
        data['Index'],
        data['Calibrated Lower'],
        data['Calibrated Upper'],
        color='lightgreen',
        alpha=0.5,
        label='Calibrated Prediction Interval'
    )

    # Plot the true labels as points and connect them with a line
    plt.plot(
        data['Index'],
        data['True Label'],
        'o-',
        color='red',
        markersize=4,
        label='Actual Value'
    )

    # Plot the predicted values as points and connect them with a line
    plt.plot(
        data['Index'],
        data['Predicted Value'],
        'o-',
        color='black',
        markersize=4,
        label='Predicted Value'
    )

    plt.xlabel('Sample Number')
    plt.ylabel(f'{TARGET_LABEL}')
    plt.title('Calibrated and Uncalibrated Prediction Intervals with True and Predicted Values')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()

    # Save the plot as an image
    image_path = 'plot.png'
    _ensure_parent_dir(image_path)
    plt.savefig(_ensure_parent_dir(image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plot
    new_sheet_name = "calibrate_and_plot_intervals"
    workbook.create_sheet(title=new_sheet_name)

    # Insert the image into the new sheet
    sheet = workbook[new_sheet_name]
    img = Image(image_path)
    sheet.add_image(img, 'A1')

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_quantile_intervals_to_excel(predictions_df, file_path, alpha=0.1):
    predictions_df = rearrange_quantiles(predictions_df, file_path)  # FIX Q2
    """
    Plots conformal prediction intervals and their coverage, and saves the plots to an Excel file.

    Parameters:
    - predictions_df: DataFrame containing prediction quantiles and actual values.
    - file_path: Path to the Excel file where plots will be saved.
    - alpha: Desired coverage level (default is 0.1).
    """
    # Extract mean prediction and uncertainty
    mean_prediction = predictions_df[0.5]
    uncertainty = predictions_df[0.95] - predictions_df[0.05]
    actual_values = predictions_df['Actual']

    # Calculate conformal scores
    conformal_scores = np.abs(mean_prediction - actual_values) / uncertainty

    # Function to find weighted quantile
    def weighted_quantile(values, quantile, sample_weight=None):
        values = np.array(values)
        if sample_weight is None:
            sample_weight = np.ones(len(values))
        sorter = np.argsort(values)
        values, sample_weight = values[sorter], sample_weight[sorter]
        weighted_quantiles = np.cumsum(sample_weight) - 0.5 * sample_weight
        weighted_quantiles /= np.sum(sample_weight)
        return np.interp(quantile, weighted_quantiles, values)

    # Calculate weighted quantile
    weights = np.ones_like(conformal_scores)  # Uniform weights for simplicity
    quantile = weighted_quantile(conformal_scores, 1 - alpha, sample_weight=weights)

    # Calculate prediction intervals
    lower_bound = mean_prediction - quantile * uncertainty
    upper_bound = mean_prediction + quantile * uncertainty

    # Naive conformal prediction
    naive_quantile = np.quantile(conformal_scores, 1 - alpha)
    naive_lower_bound = mean_prediction - naive_quantile * uncertainty
    naive_upper_bound = mean_prediction + naive_quantile * uncertainty

    # Coverage calculation (not plotted, just for potential diagnostics)
    weighted_coverage_points = (actual_values >= lower_bound) & (actual_values <= upper_bound)
    naive_coverage_points = (actual_values >= naive_lower_bound) & (actual_values <= naive_upper_bound)

    # Visualization using Matplotlib
    fig, ax = plt.subplots(figsize=(10, 5))

    x = np.arange(len(predictions_df))
    # Actual Values
    ax.plot(x, actual_values, label='Actual', color='black', linewidth=2)

    # Mean Prediction
    ax.plot(x, mean_prediction, label='Mean Prediction', color='darkorange', linewidth=2)

    # Weighted Prediction Interval (blue fill)
    ax.fill_between(x, lower_bound, upper_bound, color='blue', alpha=0.3, label='Weighted Interval')
    # Optionally, plot the bounds as lines if you want:
    # ax.plot(x, lower_bound, color='blue', linewidth=1, linestyle='--')
    # ax.plot(x, upper_bound, color='blue', linewidth=1, linestyle='--')

    # Naive Prediction Interval (red fill)
    ax.fill_between(x, naive_lower_bound, naive_upper_bound, color='red', alpha=0.3, label='Naive Interval')
    # Optionally, plot the bounds as lines:
    # ax.plot(x, naive_lower_bound, color='red', linewidth=1, linestyle='--')
    # ax.plot(x, naive_upper_bound, color='red', linewidth=1, linestyle='--')

    ax.set_title('Quantile Prediction Intervals')
    ax.set_xlabel('Sample Number')
    ax.set_ylabel(f'{TARGET_LABEL}')
    ax.grid(False)
    ax.set_facecolor('white')

    # Place the legend outside the plot
    ax.legend(loc='center left', bbox_to_anchor=(1.05, 0.5), borderaxespad=0.0)

    plt.tight_layout(rect=[0,0,0.80,1.0])  # Make space for legend

    # Save the plot to a BytesIO object
    img_data = BytesIO()
    img_data
    plt.savefig(img_data, format='png', bbox_inches='tight')
    plt.close()
    img_data.seek(0)

    # Add image to Excel
    if os.path.exists(file_path):
        wb = load_workbook(_ensure_excel_file(file_path))
        ws = wb.create_sheet(title="plot_quantile_intervals_to_excel")
    else:
        wb = Workbook()
        ws = wb.active
        ws.title = "plot_quantile_intervals_to_excel"

    img = Image(img_data)
    img.anchor = 'A1'
    ws.add_image(img)
    ws.column_dimensions['A'].width = 30
    _ensure_parent_dir(file_path)
    wb.save(_ensure_parent_dir(file_path))

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/pile-settlement-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 123.57480095537753,
  'cv_rmse': 11.116420330096263,
  'best_params': {'n_estimators': 500,
   'criterion': 'poisson',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.001},
  'test_mse': 155.90076084929467,
  'test_rmse': np.float64(12.48602261928492),
  'test_corr_coef': np.float64(0.9046621909642905),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 123.57480095537753,
  'cv_rmse': 11.116420330096263,
  'best_params': {'n_estimators': 500,
   'criterion': 'poisson',
   'max_depth': None,
   'min_samples_split': 2,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 1.0,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 0.0,
   'n_j

# **Quantile Regression with Lightgbm**

In [20]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_LGBM_df = pd.DataFrame()


# Train a model for each quantile and make predictions using LightGBM
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = LGBMRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_LGBM_df[q] = predictions

# Add actual target values to the DataFrame
predictions_LGBM_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_LGBM_df.head())

        0.05        0.1       0.15        0.2        0.3        0.4  \
0   2.215220   2.788308   3.458405   3.964871   4.361171   3.491860   
1   2.830750   3.783545   4.109049   4.255051   3.970812   3.829471   
2   1.513574  -0.318888   1.224220   1.551981   0.928140  -0.244054   
3   0.605672   0.598228   0.914446   0.902060   0.558924   0.588446   
4  43.726305  48.219541  39.362316  43.256434  44.589386  35.232989   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0   2.679314   3.331394   5.305033   3.428185   3.815665   3.495621   
1   3.753705   3.801318   4.082216   5.222433   6.078231   6.028740   
2   0.566163   1.555648   1.859668   1.652540   1.683687   1.955640   
3   0.682255   1.159323   0.539481   0.787133   0.584642   0.625511   
4  44.530045  24.885434  26.565195  29.630356  22.333443  31.381596   

        0.95    Actual  
0   6.476463  3.295000  
1   6.358814  4.595960  
2   1.777806  1.721250  
3   0.890640  0.441964  
4  44.157899  0.00000

In [21]:
quantile_regression_print(predictions_LGBM_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 76.34%


In [22]:
evaluation_features_print(predictions_LGBM_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

In [23]:
evaluate_uncertainity_matrix(predictions_LGBM_df, quantiles, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/LightGBM.xlsx", model_name='LightGBM')

Quantile crossing before sorting: 186 of 186 rows (100.0%)


{'empirical_quantiles': [np.float64(0.14516129032258066),
  np.float64(0.21505376344086022),
  np.float64(0.3010752688172043),
  np.float64(0.3763440860215054),
  np.float64(0.46236559139784944),
  np.float64(0.5268817204301075),
  np.float64(0.5860215053763441),
  np.float64(0.6290322580645161),
  np.float64(0.6774193548387096),
  np.float64(0.7043010752688172),
  np.float64(0.7903225806451613),
  np.float64(0.8548387096774194),
  np.float64(0.9086021505376344)],
 'average_interval_lengths': [np.float64(3.967250248543902),
  np.float64(8.440062148492057),
  np.float64(14.013891688448616)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.658228924670581),
 'average_crps': np.float64(4.571915268604562),
 'nll_values': array([ 1.18924939,  1.10362649,  0.57988042, -0.27567271, 15.49343926,
         1.38826775,  3.84184936,  3.83471371,  1.23506241,  2.41627459,
         0.38580998,  4.24586457, 15.68941119,  1.65265214,  2.10507302,
        -0.57534854,  1.56326115,  1.

In [24]:
calibrate_and_plot_intervals(predictions_LGBM_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

The empirical coverage before calibration is: 81.72%
The empirical coverage after calibration is: 96.77%


In [25]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_LGBM_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/LightGBM.xlsx")

# **Quantile Regression with XGBoost**

In [26]:
# Get the best hyperparameters for XGBRegressor
best_paramst = get_best_model_params(best_scores_autosampler, 'XGBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_XGB_df = pd.DataFrame()

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_paramst.copy()  # FIX Q1: was best_params (= LightGBM's settings from the cell above)

    # Adjust according to your XGBoost version
    params.update({
        'objective': 'reg:quantileerror',  # Use the correct objective
        'quantile_alpha': q,                         # Use 'alpha' to set the quantile level
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = XGBRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_XGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_XGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_XGB_df.head())

       0.05       0.1      0.15       0.2        0.3        0.4        0.5  \
0  1.227308  1.895266  2.551871  3.155832   2.872755   3.691280   4.618191   
1  1.794833  2.393334  2.677936  3.784910   4.662148   4.759102   5.310422   
2  0.729117  0.889392  1.370234  1.244741   1.705161   1.327236   1.938550   
3  0.223774  0.254526  0.124430  0.467748   1.043645   0.659630   0.754692   
4 -0.013069 -0.083866  0.590874  0.769876  14.048590  11.669280  21.338127   

         0.6        0.7        0.8       0.85        0.9       0.95    Actual  
0   2.987430   3.361517   4.081487   3.986300   5.369935   7.224391  3.295000  
1   4.160885   5.387143   5.246013   6.051652   9.876623   7.224391  4.595960  
2   2.290453   1.943999   2.362385   1.537870   2.161023   3.083071  1.721250  
3   1.137108   1.082543   1.438050   1.954305   0.577323   3.083071  0.441964  
4  31.460674  25.416048  32.849224  42.910583  45.266300  71.215759  0.000000  


In [27]:
quantile_regression_print(predictions_XGB_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 93.01%


In [28]:
evaluation_features_print(predictions_XGB_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

In [29]:
evaluate_uncertainity_matrix(predictions_XGB_df, quantiles, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/XGBoost.xlsx", model_name='XGBoost')

Quantile crossing before sorting: 183 of 186 rows (98.39%)


{'empirical_quantiles': [np.float64(0.016129032258064516),
  np.float64(0.0913978494623656),
  np.float64(0.16666666666666666),
  np.float64(0.23655913978494625),
  np.float64(0.3172043010752688),
  np.float64(0.44623655913978494),
  np.float64(0.5698924731182796),
  np.float64(0.6827956989247311),
  np.float64(0.7526881720430108),
  np.float64(0.8010752688172043),
  np.float64(0.8548387096774194),
  np.float64(0.9086021505376344),
  np.float64(0.946236559139785)],
 'average_interval_lengths': [np.float64(13.755293493950239),
  np.float64(22.15508260968472),
  np.float64(31.336125710410798)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3.079373990590749),
 'average_crps': np.float64(6.105399243576074),
 'nll_values': array([ 1.52007889,  1.81995836,  0.5844894 ,  0.87332533,  4.47967477,
         1.84453963,  4.46525288,  4.09180704,  1.81863037,  2.83036945,
         0.98978464,  5.64901283,  5.95812154,  2.15372377,  2.33279913,
         1.62704942,  1.8016555 ,  

In [30]:
calibrate_and_plot_intervals(predictions_XGB_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

The empirical coverage before calibration is: 98.92%
The empirical coverage after calibration is: 100.00%


In [31]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_XGB_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/XGBoost.xlsx")

# **Quantile Regression with GPBoost**

In [32]:
# Get the best hyperparameters for GPBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GPBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'alpha', 'quantile_alpha', 'boosting_type', 'metric', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'objective': 'quantile',
        'alpha': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = GPBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GPBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GPBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GPBoost_df.head())

       0.05       0.1       0.15        0.2        0.3        0.4        0.5  \
0  2.855688  3.232598   3.303667   3.392329   3.289176   3.616831   3.875835   
1  2.481590  3.729795   3.741389   4.311169   3.560395   4.235079   5.356411   
2  1.409760  1.267990   1.460767   1.456818   1.521633   1.349838   1.941462   
3  0.531542  0.666238   0.588726   0.800526   0.743151   0.583420   0.927923   
4  3.725951  4.882036  19.038404  23.725566  25.364863  38.638381  43.312862   

         0.6        0.7        0.8       0.85        0.9       0.95    Actual  
0   3.533569   3.503572   3.310476   3.394529   4.577168   4.155503  3.295000  
1   3.851739   0.914456   5.130174   4.718641   4.920439   6.750249  4.595960  
2   1.658858   1.856203   1.963939   1.784940   1.890570   2.120858  1.721250  
3   0.739101   1.606433   1.021035   1.185376   1.012936   1.840973  0.441964  
4  59.117519  48.752694  66.853854  74.538529  79.711095  81.073667  0.000000  


In [33]:
quantile_regression_print(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 82.80%


In [34]:
evaluation_features_print(predictions_GPBoost_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

In [35]:
evaluate_uncertainity_matrix(predictions_GPBoost_df, quantiles,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/GPBoost.xlsx", model_name='GPBoost')

Quantile crossing before sorting: 186 of 186 rows (100.0%)


{'empirical_quantiles': [np.float64(0.10752688172043011),
  np.float64(0.16129032258064516),
  np.float64(0.20967741935483872),
  np.float64(0.3010752688172043),
  np.float64(0.3817204301075269),
  np.float64(0.46236559139784944),
  np.float64(0.543010752688172),
  np.float64(0.6021505376344086),
  np.float64(0.6881720430107527),
  np.float64(0.7741935483870968),
  np.float64(0.8387096774193549),
  np.float64(0.8870967741935484),
  np.float64(0.9354838709677419)],
 'average_interval_lengths': [np.float64(9.066539139457166),
  np.float64(16.370010771129767),
  np.float64(23.01064732912685)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.377828788844018),
 'average_crps': np.float64(4.368169078272657),
 'nll_values': array([ 0.28941221,  1.51284256, -0.40205225,  0.40347271,  5.77322225,
         1.41905279,  4.18390375,  3.9674475 ,  1.29266987,  2.53955351,
         1.26006681,  4.68864836,  7.9435649 ,  2.12616951,  2.56597569,
         0.94583369,  1.70225325,  0.

In [36]:
calibrate_and_plot_intervals(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

The empirical coverage before calibration is: 87.10%
The empirical coverage after calibration is: 94.62%


In [37]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GPBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/GPBoost.xlsx")

# **Quantile Regression with NGBoost**

In [38]:
# Correct best_params assignment
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# DataFrame to store predictions
predictions_NGB_df = pd.DataFrame()

# Remove 'Score' and 'Dist' if present in best_params
for key in ['Score', 'Dist']:
    best_params.pop(key, None)

# Update params with model settings
best_params.update({
    'Dist': Normal,
    'Score': LogScore,
    'random_state': 42,
    'verbose': True
})

# Initialize and fit the NGBoost model once
model = NGBRegressor(**best_params)

# Fit the model
model.fit(X_train, y_train)

# Make predictions
pred_dist = model.pred_dist(X_test)

# Extract the mean (mu) and standard deviation (sigma) of the predicted distributions
mu = pred_dist.loc    # Mean predictions
sigma = pred_dist.scale  # Standard deviation predictions

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# For each quantile, compute predictions
for q in quantiles:
    # Calculate the z-score for the desired quantile
    z = norm.ppf(q)

    # Compute the quantile prediction per sample
    predictions = mu + z * sigma

    # Add predictions to the DataFrame
    predictions_NGB_df[q] = predictions

# Add actual target values to the DataFrame
predictions_NGB_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_NGB_df.head())

[iter 0] loss=4.6028 val_loss=0.0000 scale=2.0000 norm=32.2242


[iter 100] loss=2.4454 val_loss=0.0000 scale=0.5000 norm=1.8386


[iter 200] loss=2.1477 val_loss=0.0000 scale=0.2500 norm=0.7238


[iter 300] loss=1.9281 val_loss=0.0000 scale=0.0002 norm=0.0006


[iter 400] loss=1.8420 val_loss=0.0000 scale=0.5000 norm=1.1314


[iter 500] loss=1.7433 val_loss=0.0000 scale=0.1250 norm=0.2621


[iter 600] loss=1.7067 val_loss=0.0000 scale=0.5000 norm=0.9961


[iter 700] loss=1.5921 val_loss=0.0000 scale=1.0000 norm=1.7758


[iter 800] loss=1.5081 val_loss=0.0000 scale=0.5000 norm=0.8365


[iter 900] loss=1.4816 val_loss=0.0000 scale=0.2500 norm=0.4055


        0.05        0.1       0.15        0.2        0.3        0.4  \
0   3.010113   3.069651   3.109820   3.141746   3.193732   3.238152   
1   2.134443   2.508853   2.761466   2.962235   3.289155   3.568496   
2   1.572950   1.652177   1.705631   1.748114   1.817292   1.876401   
3  -0.930654  -0.751403  -0.630464  -0.534344  -0.377830  -0.244094   
4  23.533496  24.977379  25.951560  26.725808  27.986548  29.063804   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0   3.279670   3.321188   3.365608   3.417594   3.449520   3.489690   
1   3.829590   4.090683   4.370024   4.696944   4.897713   5.150326   
2   1.931650   1.986898   2.046008   2.115185   2.157668   2.211122   
3  -0.119094   0.005906   0.139642   0.296157   0.392276   0.513216   
4  30.070690  31.077575  32.154831  33.415571  34.189820  35.164001   

        0.95    Actual  
0   3.549227  3.295000  
1   5.524736  4.595960  
2   2.290349  1.721250  
3   0.692467  0.441964  
4  36.607883  0.00000

In [39]:
quantile_regression_print(predictions_NGB_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 58.06%


In [40]:
evaluation_features_print(predictions_NGB_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

In [41]:
evaluate_uncertainity_matrix(predictions_NGB_df, quantiles,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/NGBoost.xlsx", model_name='NGBoost')

Quantile crossing before sorting: 0 of 186 rows (0.0%)


{'empirical_quantiles': [np.float64(0.1935483870967742),
  np.float64(0.25268817204301075),
  np.float64(0.3172043010752688),
  np.float64(0.3548387096774194),
  np.float64(0.44086021505376344),
  np.float64(0.5053763440860215),
  np.float64(0.521505376344086),
  np.float64(0.5591397849462365),
  np.float64(0.5967741935483871),
  np.float64(0.6344086021505376),
  np.float64(0.6774193548387096),
  np.float64(0.7365591397849462),
  np.float64(0.7741935483870968)],
 'average_interval_lengths': [np.float64(2.62494100615055),
  np.float64(3.9970441829436254),
  np.float64(5.1301506690495335)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(19.89212528944589),
 'average_crps': np.float64(5.731279117997623),
 'nll_values': array([-8.85312568e-01,  1.22555139e+00, -1.38555269e-01,  8.59033906e-01,
         3.09226794e+01,  2.22857857e-01,  2.02084469e+00,  9.34071586e+01,
         3.89382587e+00,  1.58166547e+00,  6.27482393e-01,  2.48279432e+01,
         1.82150993e+02,  2.08

In [42]:
calibrate_and_plot_intervals(predictions_NGB_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

The empirical coverage before calibration is: 61.29%
The empirical coverage after calibration is: 95.70%


In [43]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_NGB_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/NGBoost.xlsx")

# **Quantile Regression with Gradient Boosting**

In [44]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_GBR_df = pd.DataFrame()

# Custom quantile loss function
def quantile_loss(q, y_true, y_pred):
    e = y_true - y_pred
    return np.mean(np.maximum(q * e, (q - 1) * e))

# Train a model for each quantile and make predictions using GradientBoostingRegressor
for q in quantiles:

    params = best_params.copy()

    params.update({
        'loss': 'quantile',
        'alpha': q,
        'random_state': 42
    })
    # Initialize the model with quantile-specific settings
    model = GradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_GBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_GBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_GBR_df.head())

   0.05       0.1      0.15       0.2        0.3        0.4        0.5  \
0   0.0  3.186646  5.319993  3.353342   3.129051   3.262272   2.874027   
1   0.0  3.122376  3.866180  4.188658   4.391441   3.536875   3.835749   
2   0.0  1.199814  1.286655  1.459475   1.869118   1.801276   1.737316   
3   0.0  0.530820  0.452280  0.639228   0.272131   0.627118   0.082588   
4   0.0  2.845298  5.790272  4.258517  11.193352  16.426116  17.788734   

         0.6        0.7        0.8       0.85        0.9       0.95    Actual  
0   3.272076   2.188722   2.490895   3.711268   4.362802  11.865071  3.295000  
1   4.223545   5.218387   4.672230   4.079104   5.627977  12.120658  4.595960  
2   1.464713   1.939963   1.980702   1.802901   1.770323   2.058382  1.721250  
3   0.566611   0.474635   1.152407   1.216531   0.784481   1.960988  0.441964  
4  32.968787  18.864387  45.866889  41.011046  54.562903  57.406563  0.000000  


In [45]:
quantile_regression_print(predictions_GBR_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 93.01%


In [46]:
evaluation_features_print(predictions_GBR_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

In [47]:
evaluate_uncertainity_matrix(predictions_GBR_df, quantiles,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx", model_name='GBM')

Quantile crossing before sorting: 186 of 186 rows (100.0%)


{'empirical_quantiles': [np.float64(0.016129032258064516),
  np.float64(0.11290322580645161),
  np.float64(0.1881720430107527),
  np.float64(0.2903225806451613),
  np.float64(0.3817204301075269),
  np.float64(0.46774193548387094),
  np.float64(0.5806451612903226),
  np.float64(0.6397849462365591),
  np.float64(0.7150537634408602),
  np.float64(0.8064516129032258),
  np.float64(0.8655913978494624),
  np.float64(0.8924731182795699),
  np.float64(0.9301075268817204)],
 'average_interval_lengths': [np.float64(8.359201528216191),
  np.float64(15.244344877547505),
  np.float64(25.08947905971822)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.5891001179770816),
 'average_crps': np.float64(4.292867662112722),
 'nll_values': array([2.20178001, 2.22916158, 0.45313594, 0.42345103, 4.29787746,
        1.63962848, 4.30157952, 3.50085237, 1.60814815, 2.64227066,
        0.40828089, 4.18237925, 9.20104641, 1.91007932, 2.28642586,
        0.30952505, 1.43432606, 0.02440923, 2.1232

In [48]:
calibrate_and_plot_intervals(predictions_GBR_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

The empirical coverage before calibration is: 97.85%
The empirical coverage after calibration is: 97.85%


In [49]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_GBR_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/Gradient Boosting.xlsx")

# **Quantile Regression with CatBoost**

In [50]:
# Get the best hyperparameters for CatBoostRegressor
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_CatBoost_df = pd.DataFrame()

# Parameters that may conflict with quantile-specific settings
incompatible_keys = ['objective', 'loss_function', 'eval_metric']

# Train a model for each quantile and make predictions
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Remove incompatible parameters
    for key in incompatible_keys:
        params.pop(key, None)

    # Update params with quantile-specific settings
    params.update({
        'loss_function': 'Quantile:alpha={}'.format(q),
        'random_seed': 42
    })

    # Initialize the model with the updated parameters
    model = CatBoostRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train, verbose=0)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_CatBoost_df[q] = predictions

# Add actual target values to the DataFrame
predictions_CatBoost_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_CatBoost_df.head())

       0.05       0.1      0.15        0.2        0.3        0.4        0.5  \
0  3.133023  2.998196  2.807449   3.255507   3.360722   3.329821   2.970394   
1  2.612325  3.380214  3.893793   3.526528   4.122475   4.637939   4.172483   
2  1.419787  1.331921  1.557428   1.567404   1.663683   1.789245   1.846826   
3  0.452136  0.474609  0.652099   0.600328   0.698006   0.614088   0.664280   
4  7.793064  2.525542  0.007899  14.734984  13.576367  15.723727  23.686933   

         0.6        0.7        0.8       0.85        0.9       0.95    Actual  
0   2.113828   2.791870   3.447912   3.781796   4.369189  16.934384  3.295000  
1   3.911694   5.920502   5.921618   6.720843  11.788464  13.354771  4.595960  
2   1.918336   1.763730   1.818489   2.087961   2.141893   2.861464  1.721250  
3   0.651663   0.612181   0.410800   0.465364   0.447760   0.207568  0.441964  
4  28.939761  26.245512  21.786591  27.093245  31.404273  54.621054  0.000000  


In [51]:
quantile_regression_print(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 78.49%


In [52]:
evaluation_features_print(predictions_CatBoost_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

In [53]:
evaluate_uncertainity_matrix(predictions_CatBoost_df, quantiles,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/CatBoost.xlsx", model_name='CatBoost')

Quantile crossing before sorting: 185 of 186 rows (99.46%)


{'empirical_quantiles': [np.float64(0.13440860215053763),
  np.float64(0.23118279569892472),
  np.float64(0.26881720430107525),
  np.float64(0.3279569892473118),
  np.float64(0.41935483870967744),
  np.float64(0.489247311827957),
  np.float64(0.532258064516129),
  np.float64(0.6451612903225806),
  np.float64(0.7311827956989247),
  np.float64(0.8064516129032258),
  np.float64(0.8494623655913979),
  np.float64(0.8817204301075269),
  np.float64(0.9193548387096774)],
 'average_interval_lengths': [np.float64(8.088034601165203),
  np.float64(13.747643955892462),
  np.float64(21.520430476338255)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.8342884104052413),
 'average_crps': np.float64(4.709781885142283),
 'nll_values': array([ 2.42419357,  2.11075169,  0.1638026 , -0.42011951,  4.58954438,
         2.02505188,  3.73305567,  3.33367201,  1.93558512,  3.05985938,
        -0.41787426,  4.20078012, 11.89829373,  2.36779109,  2.85881713,
        -0.36208565,  1.66331226,  2

In [54]:
calibrate_and_plot_intervals(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

The empirical coverage before calibration is: 83.87%
The empirical coverage after calibration is: 97.85%


In [55]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_CatBoost_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/CatBoost.xlsx")

# **Quantile Regression with HistGradientBoosting**

In [56]:

# Assume best_params is a dictionary containing the best parameters for HistGradientBoostingRegressor
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Dictionary to store models for each quantile
models_quantile = {}

# DataFrame to store predictions
predictions_HGBR_df = pd.DataFrame()

# Train a model for each quantile and make predictions using HistGradientBoostingRegressor
for q in quantiles:
    # Create a copy of best_params to avoid modifying the original
    params = best_params.copy()

    # Update params with quantile-specific settings
    params.update({
        'loss': 'quantile',
        'quantile': q,
        'random_state': 42
    })

    # Initialize the model with the updated parameters
    model = HistGradientBoostingRegressor(**params)

    # Fit the model
    model.fit(X_train, y_train)

    # Store the model
    models_quantile[q] = model

    # Make predictions
    predictions = model.predict(X_test)

    # Add predictions to the DataFrame
    predictions_HGBR_df[q] = predictions

# Add actual target values to the DataFrame
predictions_HGBR_df['Actual'] = y_test.values.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_HGBR_df.head())

       0.05           0.1          0.15           0.2       0.3       0.4  \
0  3.280753  4.055408e+00  4.065277e+00  2.529884e+00  1.706416  2.928852   
1  4.718614  3.612184e+00  3.869183e+00  4.027676e+00  4.082738  4.510623   
2  1.711202  1.493224e+00  1.269733e+00  9.437985e-01  0.402978  1.558759   
3  0.560955  6.441116e-01  5.181146e-01  6.047362e-01  0.103321 -0.301046   
4  3.564170  1.427890e-36  3.221821e-36  4.954725e-36  0.917519  0.758759   

        0.5       0.6       0.7        0.8       0.85        0.9       0.95  \
0  0.482300  3.509550  4.215797   3.290104   4.886450   3.029195   3.796670   
1  4.155317  4.941475  4.787055   3.628988   5.154257   5.016741   3.313016   
2  1.476293  1.713140  1.439986   1.686271   1.963297   2.261469   2.047580   
3  0.396500  0.465712  0.322421   0.550329   0.578348   1.150361   0.783516   
4  1.155005  9.604607  9.632233  10.664132  27.340736  47.258477  39.957971   

     Actual  
0  3.295000  
1  4.595960  
2  1.721250  
3  0.4

In [57]:
quantile_regression_print(predictions_HGBR_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/HGBM.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 84.41%


In [58]:
evaluation_features_print(predictions_HGBR_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/HGBM.xlsx")

In [59]:
evaluate_uncertainity_matrix(predictions_HGBR_df, quantiles,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/HGBM.xlsx", model_name='HGBM')

Quantile crossing before sorting: 185 of 186 rows (99.46%)


{'empirical_quantiles': [np.float64(0.08602150537634409),
  np.float64(0.1827956989247312),
  np.float64(0.25806451612903225),
  np.float64(0.3279569892473118),
  np.float64(0.3924731182795699),
  np.float64(0.46236559139784944),
  np.float64(0.521505376344086),
  np.float64(0.6344086021505376),
  np.float64(0.6989247311827957),
  np.float64(0.7688172043010753),
  np.float64(0.8172043010752689),
  np.float64(0.8870967741935484),
  np.float64(0.9301075268817204)],
 'average_interval_lengths': [np.float64(6.921500631843271),
  np.float64(12.774311099315275),
  np.float64(17.41673844095397)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(2.2326676188106176),
 'average_crps': np.float64(4.337383008400265),
 'nll_values': array([ 1.21069395e+00,  6.48490518e-01,  3.89268936e-01,  1.30837398e-01,
         3.61455011e+00,  1.35652189e+00,  3.77953318e+00,  3.98704291e+00,
         1.10659029e+00,  2.44372896e+00,  1.57560754e-02,  3.98360340e+00,
         1.04323979e+01,  1.

In [60]:
calibrate_and_plot_intervals(predictions_HGBR_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/HGBM.xlsx")

The empirical coverage before calibration is: 87.10%
The empirical coverage after calibration is: 96.77%


In [61]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_HGBR_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/HGBM.xlsx")

# **Quantile Regression with TabNet**

In [62]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')

# Quantiles
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# Storage
models_quantile = {}
predictions_TabNet_df = pd.DataFrame()

# Convert data to numpy (TabNet requirement)
X_train_np = np.array(X_train)
y_train_np = np.array(y_train).reshape(-1, 1)
X_test_np  = np.array(X_test)

# Quantile loss (pinball loss)
def quantile_loss(q):
    def loss(y_pred, y_true):
        error = y_true - y_pred
        return torch.mean(torch.maximum(q * error, (q - 1) * error))
    return loss

# FIX Q3: early stopping on a 20% validation split of TRAIN, scored with the pinball loss of
# the quantile being trained (was: eval_set = the training data itself, scored with MSE,
# so early stopping never triggered).
from pytorch_tabnet.metrics import Metric
X_fit_np, X_val_np, y_fit_np, y_val_np = train_test_split(X_train_np, y_train_np, test_size=0.2, random_state=42)

def pinball_metric(q):
    class Pinball(Metric):
        def __init__(self):
            self._name = f'pinball_{q}'
            self._maximize = False
        def __call__(self, y_true, y_score, weights=None):  # FIX B8: pytorch-tabnet2 also passes sample weights
            e = np.asarray(y_true, dtype=float) - np.asarray(y_score, dtype=float)
            return float(np.mean(np.maximum(q * e, (q - 1) * e)))
    return Pinball

# Train one TabNet per quantile
for q in quantiles:
    model = TabNetRegressor(
        **best_params,
    )

    model.fit(
        X_fit_np,
        y_fit_np,
        eval_set=[(X_val_np, y_val_np)],
        eval_metric=[pinball_metric(q)],
        loss_fn=quantile_loss(q),
        max_epochs=200,
        patience=30,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False
    )

    # Store model
    models_quantile[q] = model

    # Predict
    preds = model.predict(X_test_np).ravel()
    predictions_TabNet_df[q] = preds

# Add actual values
predictions_TabNet_df['Actual'] = y_test.values.ravel()

print(predictions_TabNet_df.head())


Early stopping occurred at epoch 138 with best_epoch = 108 and best_val_0_pinball_0.05 = 0.25427



Early stopping occurred at epoch 169 with best_epoch = 139 and best_val_0_pinball_0.1 = 0.25453



Early stopping occurred at epoch 175 with best_epoch = 145 and best_val_0_pinball_0.15 = 0.40402



Early stopping occurred at epoch 173 with best_epoch = 143 and best_val_0_pinball_0.2 = 0.46424



Early stopping occurred at epoch 66 with best_epoch = 36 and best_val_0_pinball_0.3 = 0.98635



Early stopping occurred at epoch 116 with best_epoch = 86 and best_val_0_pinball_0.4 = 0.78984


... (3 outputs left out)


Early stopping occurred at epoch 98 with best_epoch = 68 and best_val_0_pinball_0.8 = 0.90789



Early stopping occurred at epoch 101 with best_epoch = 71 and best_val_0_pinball_0.85 = 0.56858



Early stopping occurred at epoch 75 with best_epoch = 45 and best_val_0_pinball_0.9 = 1.05596



Early stopping occurred at epoch 168 with best_epoch = 138 and best_val_0_pinball_0.95 = 0.24728
       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0  2.564377  2.955679  3.323689  3.100513  3.099074  3.325306  3.911108   
1  3.668575  4.008474  4.025150  3.619225  3.823653  4.257343  3.976180   
2  1.444344  1.599094  1.519374  1.791106  1.889538  1.812440  1.655594   
3  0.577743  0.518192  0.660765  0.623773  0.696122  0.547121  0.954947   
4 -0.970679 -0.600982  1.912901  0.108221  3.512913  1.325527  0.203371   

        0.6       0.7       0.8      0.85       0.9      0.95    Actual  
0  3.593581  3.589516  6.025471  4.810397  5.012496  5.377497  3.295000  
1  3.808817  4.502807  5.007007  4.422565  6.315056  5.971889  4.595960  
2  1.980571  2.169521  2.347188  2.486588  2.585648  3.069601  1.721250  
3  0.605489  0.838125  1.138292  1.608889  1.934305  1.419369  0.441964  
4  0.792487 -2.590580  0.201252  1.727439  0.003633  4.526287  0.000000  


In [63]:
quantile_regression_print(predictions_TabNet_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/TabNet.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 95.16%


In [64]:
evaluation_features_print(predictions_TabNet_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/TabNet.xlsx")

In [65]:
evaluate_uncertainity_matrix(predictions_TabNet_df, quantiles, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/TabNet.xlsx", model_name='TabNet')

Quantile crossing before sorting: 186 of 186 rows (100.0%)


{'empirical_quantiles': [np.float64(0.021505376344086023),
  np.float64(0.06989247311827956),
  np.float64(0.11827956989247312),
  np.float64(0.20967741935483872),
  np.float64(0.25806451612903225),
  np.float64(0.3924731182795699),
  np.float64(0.5),
  np.float64(0.6290322580645161),
  np.float64(0.7419354838709677),
  np.float64(0.8279569892473119),
  np.float64(0.9032258064516129),
  np.float64(0.956989247311828),
  np.float64(0.9731182795698925)],
 'average_interval_lengths': [np.float64(6.006979303654804),
  np.float64(11.260731186116896),
  np.float64(16.528980444515906)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(1.946667889499349),
 'average_crps': np.float64(2.371942390103969),
 'nll_values': array([ 1.00890557,  0.96244185,  0.27182186,  0.25035488,  1.69502617,
         1.09519727,  3.68208595,  3.97702996,  0.72498598,  2.14059159,
         0.17729213,  4.15206877,  4.84076859,  1.13215346,  2.1614639 ,
        -0.18047916,  0.95416602,  0.60905769,  1

In [66]:
calibrate_and_plot_intervals(predictions_TabNet_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/TabNet.xlsx")

The empirical coverage before calibration is: 95.70%
The empirical coverage after calibration is: 96.77%


In [67]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_TabNet_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/TabNet.xlsx")

# **Quantile Regression with PGBM**

In [68]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [69]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())

Training on CPU
Estimator 0/300, Train metric: 22.1366


Estimator 1/300, Train metric: 20.8612
Estimator 2/300, Train metric: 19.7222
Estimator 3/300, Train metric: 18.3465
Estimator 4/300, Train metric: 17.2399
Estimator 5/300, Train metric: 16.3585
Estimator 6/300, Train metric: 15.5218
Estimator 7/300, Train metric: 14.6674
Estimator 8/300, Train metric: 14.0307
Estimator 9/300, Train metric: 13.4829
Estimator 10/300, Train metric: 12.9849
Estimator 11/300, Train metric: 12.5348
Estimator 12/300, Train metric: 12.0420
Estimator 13/300, Train metric: 11.6170


Estimator 14/300, Train metric: 11.1625
Estimator 15/300, Train metric: 10.7936
Estimator 16/300, Train metric: 10.4034
Estimator 17/300, Train metric: 10.0656
Estimator 18/300, Train metric: 9.7242
Estimator 19/300, Train metric: 9.4833
Estimator 20/300, Train metric: 9.1477
Estimator 21/300, Train metric: 8.9779


Estimator 22/300, Train metric: 8.7714
Estimator 23/300, Train metric: 8.5725
Estimator 24/300, Train metric: 8.3745
Estimator 25/300, Train metric: 8.1820
Estimator 26/300, Train metric: 8.0159
Estimator 27/300, Train metric: 7.8346
Estimator 28/300, Train metric: 7.6656


Estimator 29/300, Train metric: 7.4570
Estimator 30/300, Train metric: 7.3069
Estimator 31/300, Train metric: 7.1485
Estimator 32/300, Train metric: 6.9983
Estimator 33/300, Train metric: 6.8945
Estimator 34/300, Train metric: 6.7534
Estimator 35/300, Train metric: 6.6278


Estimator 36/300, Train metric: 6.5443
Estimator 37/300, Train metric: 6.4147
Estimator 38/300, Train metric: 6.3123
Estimator 39/300, Train metric: 6.2010
Estimator 40/300, Train metric: 6.1134
Estimator 41/300, Train metric: 6.0387
Estimator 42/300, Train metric: 5.9352


... (35 outputs left out)

Estimator 284/300, Train metric: 1.7951
Estimator 285/300, Train metric: 1.7929
Estimator 286/300, Train metric: 1.7863
Estimator 287/300, Train metric: 1.7833
Estimator 288/300, Train metric: 1.7811
Estimator 289/300, Train metric: 1.7799
Estimator 290/300, Train metric: 1.7770


Estimator 291/300, Train metric: 1.7717
Estimator 292/300, Train metric: 1.7697
Estimator 293/300, Train metric: 1.7691
Estimator 294/300, Train metric: 1.7663
Estimator 295/300, Train metric: 1.7634
Estimator 296/300, Train metric: 1.7625
Estimator 297/300, Train metric: 1.7595


Estimator 298/300, Train metric: 1.7573
Estimator 299/300, Train metric: 1.7558


        0.05        0.1       0.15        0.2        0.3        0.4  \
0   3.265267   3.274162   3.278327   3.282201   3.285167   3.289589   
1   2.858373   2.873633   2.881102   2.885765   2.895033   2.899088   
2   1.759889   1.765304   1.767196   1.770087   1.771839   1.774391   
3   0.728211   0.761958   0.777914   0.783758   0.797529   0.805729   
4  10.463576  10.919612  11.347294  11.522360  11.710670  11.904534   

         0.5        0.6        0.7        0.8       0.85        0.9  \
0   3.293096   3.297314   3.300866   3.308504   3.310358   3.313658   
1   2.905445   2.912283   2.917895   2.925975   2.930824   2.937810   
2   1.775692   1.777804   1.779997   1.782661   1.784653   1.786698   
3   0.816341   0.823775   0.834706   0.846454   0.855265   0.866185   
4  12.014844  12.160096  12.234157  12.404488  12.610868  12.860250   

        0.95    Actual  
0   3.320782  3.295000  
1   2.942596  4.595960  
2   1.793037  1.721250  
3   0.888910  0.441964  
4  13.173440  0.00000

In [70]:
quantile_regression_print(predictions_PGBM_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/PGBM.xlsx")

First few actual values: [3.295    4.59596  1.72125  0.441964 0.      ]
Coverage of 90% prediction interval: 6.45%


In [71]:
evaluation_features_print(predictions_PGBM_df,x_test,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/PGBM.xlsx")

In [72]:
evaluate_uncertainity_matrix(predictions_PGBM_df, quantiles, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/PGBM.xlsx", model_name='PGBM')

Quantile crossing before sorting: 0 of 186 rows (0.0%)


{'empirical_quantiles': [np.float64(0.510752688172043),
  np.float64(0.510752688172043),
  np.float64(0.510752688172043),
  np.float64(0.5161290322580645),
  np.float64(0.521505376344086),
  np.float64(0.532258064516129),
  np.float64(0.532258064516129),
  np.float64(0.5376344086021505),
  np.float64(0.543010752688172),
  np.float64(0.5483870967741935),
  np.float64(0.553763440860215),
  np.float64(0.5591397849462365),
  np.float64(0.5752688172043011)],
 'average_interval_lengths': [np.float64(0.2167417957217142),
  np.float64(0.3538248784719936),
  np.float64(0.5105001804849474)],
 'coverage_levels': [0.6, 0.8, 0.9],
 'average_nll': np.float64(3676.6772302781465),
 'average_crps': np.float64(4.609647773759433),
 'nll_values': array([-3.15659237e+00,  2.17725065e+03,  1.09169974e+01,  2.72679712e+01,
         1.07096688e+02,  1.83088114e+02,  5.78146418e+01,  4.53019684e+03,
         4.38204924e+02,  5.26209478e+02,  1.42788945e+02,  5.58388226e+03,
         1.51985880e+04,  1.72190748

In [73]:
calibrate_and_plot_intervals(predictions_PGBM_df, "./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/PGBM.xlsx")

The empirical coverage before calibration is: 6.45%
The empirical coverage after calibration is: 100.00%


In [74]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

plot_quantile_intervals_to_excel(predictions_PGBM_df,"./drive/MyDrive/Regression/pile_settlement_uncertainty_analysis/quantile_regression/PGBM.xlsx")